# 01 · F-O1 raw tapes → inventory → spliced composite (v2)
**F-O gas field, Bredasdorp Basin — Notebook 01 (replaces the first draft)**

Version 2 is tested on your actual F-O1 tapes. Three things changed from v1:
1. **Tape repair.** Your `*.tlis` files are LIS wrapped in TIF, with 2 padding bytes after most records.
   `dlisio` stops at the first padded record, so `petrotools.tif_repair` rewrites each tape as clean LIS first
   (the originals are untouched; repaired copies go to `~/.fo_repaired_lis`).
2. **Inventory of every frame** in every tape (run, depth range, curves).
3. **Spliced composite** for F-O1 (3 logging runs, 361–3928 m) exported as CSV + LAS for IP, with a provenance table.

**Setup:** keep the `petrotools` folder next to this notebook. Edit only Cell 2.

In [ ]:
# Cell 1 — install (once), then restart the kernel
%pip install dlisio lasio matplotlib pandas openpyxl

In [ ]:
# Cell 2 — paths: EDIT RAW_DIR (copy the folder path from File Explorer's address bar)
from pathlib import Path
WELL    = "F-O1"
RAW_DIR = Path(r"C:\Users\YOUR_NAME\...\Wells Data_Bredasdorp\F-O1\Raw_logs")   # <-- edit
OUT_DIR = RAW_DIR.parent / "Processed"
OUT_DIR.mkdir(exist_ok=True)
raw_files = sorted(p for p in RAW_DIR.iterdir() if p.is_file())
print(len(raw_files), "raw files")

## Step 1 — inventory every frame on every tape

In [ ]:
# Cell 3
import logging, warnings, numpy as np, pandas as pd
logging.getLogger("dlisio").setLevel(logging.CRITICAL); warnings.filterwarnings("ignore")
from petrotools.logio import lis_frames
rows = []
for p in raw_files:
    try:
        for key, df, units in lis_frames(str(p), WELL):
            if len(df) < 2: continue
            ok = np.isfinite(df.index)
            rows.append(dict(file=p.name, frame=key, top_m=round(df.index[ok].min(), 2), base_m=round(df.index[ok].max(), 2),
                             step_m=round(float(np.median(np.diff(df.index[ok]))), 4), n=len(df), n_curves=df.shape[1],
                             curves=" ".join(df.columns)))
    except Exception as e:
        rows.append(dict(file=p.name, frame="ERROR", curves=str(e)))
inv = pd.DataFrame(rows)
inv.to_csv(OUT_DIR / f"{WELL}_frame_inventory.csv", index=False)
pd.set_option("display.max_colwidth", 90)
inv[["file", "frame", "top_m", "base_m", "step_m", "n", "n_curves"]]

**How to read the inventory (F-O1):** frames with a sensible depth range are logging passes. Tapes `re175c/e/o` are the
well-seismic (checkshot) survey and `re175i` holds the 16 RFT pressure tests (their index is time, not depth), so they
show tiny or odd depth ranges; they are handled in later notebooks. The three main logging runs are:

| Run | Depth (m) | Tool string | Tape used |
|---|---|---|---|
| 1 | 362–1515 | DIL-BHC-GR | re175b |
| 2 | 1494–3109 | DIL-BHC-GR + LDT | re175g, re175q |
| 3 | 3089–3928 | DIL-BHC-GR, DLL-LDT-CNL | re175n, re175j |

## Step 2 — splice the composite

In [ ]:
# Cell 4 — segment table: (tape pattern, frame, {source curve: output name}, top m, base m). Deeper runs take priority.
from petrotools.composite import build
import lasio
std = {"GR": "GR", "SP": "SP", "CALS": "CALS", "ILD": "ILD", "ILM": "ILM", "SFLU": "SFLU", "DT": "DT"}
SEGMENTS = [
    ("re175n.*", "L0D1", {**std, "DTL": "DTL"}, 3088.7, 3927.7),
    ("re175j.*", "L0D1", {"LLD": "LLD", "LLS": "LLS", "RHOB": "RHOB", "DRHO": "DRHO", "NPHI": "NPHI",
                          "TNPH": "TNPH", "NPOR": "NPOR", "PEF": "PEF", "CALI": "CALI"}, 3093.6, 3915.8),
    ("re175n.*", "L3D1", {"RHOB": "RHOB_ALT"}, 3092.0, 3915.8),
    ("re175g.*", "L0D1", std, 1494.3, 3088.6),
    ("re175q.*", "L1D1", {"RHOB": "RHOB", "DRHO": "DRHO", "PEF": "PEF", "CALI": "CALI"}, 1495.3, 3093.5),
    ("re175b.*", "L0D1", std, 361.6, 1494.2),
]
comp, units, prov = build(RAW_DIR, WELL, SEGMENTS, top=361.6, base=3927.7)
comp.to_csv(OUT_DIR / f"{WELL}_composite_v0.csv", float_format="%.4f")
prov.to_csv(OUT_DIR / f"{WELL}_composite_provenance.csv", index=False)
las = lasio.LASFile()
las.well["WELL"].value, las.well["FLD"].value, las.well["CTRY"].value = WELL, "F-O Gas Field", "South Africa"
las.other = "Composite v0 from raw LIS tapes. Logger depth, m below KB, no core shift. RHOB_ALT = field copy of run-3 density."
las.append_curve("DEPT", comp.index.values, unit="M")
for c in comp.columns: las.append_curve(c, comp[c].values, unit=units.get(c, ""))
las.write(str(OUT_DIR / f"{WELL}_composite_v0.las"), version=2.0, fmt="%.4f")
comp.loc[3695:3815].describe().T[["count", "mean", "50%", "min", "max"]].round(3)

**⚠ Open QC item — two density curves.** Run 3 exists twice on tape: `re175j` (file dated 17 Sep 1989) and `re175n`
(field copy, 12 Sep 1989). NPHI, LLD and the rest are identical, but RHOB differs by a near-constant **0.085 g/cm³**
(r = 0.998, no depth lag), i.e. about 5 porosity units. `RHOB` = re175j, `RHOB_ALT` = re175n. Do not pick one by eye:
Notebook 02 will decide against core porosity (F-O1 cores 1–2, log depth = core depth + 3.5 m per RPS 2008).

## Step 3 — quick-look over the USM reservoir

In [ ]:
# Cell 5
from petrotools.qcplot import qc_plot
from IPython.display import Image
markers = {3695.0: "M10 BLVS", 3702.5: "M8", 3715.0: "M7", 3739.5: "M6", 3750.5: "M5",
           3771.5: "M4", 3792.5: "M3", 3811.5: "M2", 3815.5: "M1"}          # RPS (2008) Table 1, F-O1
bands = [(3709.63, 3715.71, "Low-K unit", "#eda100")]                         # Chemostrat Table of Tops, F-O1
cores = [(3704 + 3.5, 3722.04 + 3.5), (3722.34 + 3.5, 3727.87 + 3.5)]         # cores 1-2 shifted to log depth
png = OUT_DIR / f"{WELL}_USM_QC.png"
qc_plot(comp, png, f"{WELL} · USM reservoir · composite v0", 3680, 3830, markers, bands, cores)
Image(filename=str(png))

**What to look for:** the upper USM sand (≈3702–3740 m) has low GR, lower density, higher deep resistivity and
neutron–density gas crossover; the lower USM (≈3740–3800 m) is tight (RHOB ≈ 2.6). That matches the RPS (2008)
model of secondary, telodiagenetic porosity confined to the top of the USM. PEF ≈ 4–5 in clean sand is too high for
quartz (≈1.8) — likely barite in the mud; treat PEF with care in the mineral solver.

**Next (Notebook 02):** digitise F-O1 core poro-perm, apply the core shift, resolve RHOB vs RHOB_ALT, and set up
environmental QC (caliper/DRHO flags). Then repeat Notebook 01 for F-O2…F-S1 (only the SEGMENTS table changes).